In [1]:
def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    LOOKBACK_DAYS = 7
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)

    sql = f"""
    WITH cte_bar1m AS (
        SELECT
            date, instrument, volume,
            strftime(date, '%Y-%m-%d') as trading_day,
            (ask_price1 + bid_price1) / 2 as mid_price,
            (
                COALESCE(bid_volume1, 0) * 1.0 +
                COALESCE(bid_volume2, 0) * EXP(-0.3) +
                COALESCE(bid_volume3, 0) * EXP(-0.6) +
                COALESCE(bid_volume4, 0) * EXP(-0.9) +
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) as weight_bid,
            (
                COALESCE(ask_volume1, 0) * 1.0 +
                COALESCE(ask_volume2, 0) * EXP(-0.3) +
                COALESCE(ask_volume3, 0) * EXP(-0.6) +
                COALESCE(ask_volume4, 0) * EXP(-0.9) +
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) as weight_ask,
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) as weighted_imbalance,
            (ask_price1 - bid_price1) / mid_price as relative_spread,
            (
                (COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0))
                - (COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0))
            ) / (
                (COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0))
                + (COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0))
                + 1e-8
            ) as depth_ratio,
            (weighted_imbalance / (sqrt(abs(relative_spread)) + 1e-8)) * weighted_imbalance as raw_pressure
        FROM {bar1m}
        WHERE ask_price1 > 0 AND bid_price1 > 0
    ),
    cte_rolling AS (
        SELECT *,
            lag(mid_price, 1) OVER (PARTITION BY instrument, trading_day ORDER BY date) as prev_mid_price,
            lag(weighted_imbalance, 1) OVER (PARTITION BY instrument, trading_day ORDER BY date) as prev_imb,
            abs(mid_price / prev_mid_price - 1) as returns,
            CASE WHEN mid_price > 0 AND prev_mid_price > 0
                THEN log(mid_price / prev_mid_price)
                ELSE NULL END as log_ret
        FROM cte_bar1m
    ),
    cte_window AS (
        SELECT
            trading_day, instrument,

            first(mid_price ORDER BY date) as open_p,
            max(mid_price) as high_p,
            min(mid_price) as low_p,
            last(mid_price ORDER BY date) as close_p,
            SUM(volume) as total_volume,

            -- 模板原版: 压力因子
            avg(raw_pressure) as raw_pressure_mean,
            nanstd(raw_pressure) as raw_pressure_std,
            (last(raw_pressure ORDER BY date) - raw_pressure_mean) / raw_pressure_std as standardized_pressure,

            -- 波动率阈值 (模板原版)
            CASE
                WHEN COUNT(*) > 10
                THEN quantile(returns, 0.8)
                ELSE 0.01
            END as volatility_threshold,

            -- 日内波动
            nanstd(log_ret) as volatility,

            -- 波动调整后压力 (模板原版)
            CASE
                WHEN volatility > volatility_threshold
                THEN standardized_pressure * 0.7
                ELSE standardized_pressure
            END as adjusted_pressure,

            -- 微观结构辅助
            avg(weighted_imbalance) as imb_mean,
            last(weighted_imbalance ORDER BY date) as imb_close,
            nanstd(weighted_imbalance) as imb_std,
            avg(relative_spread) as spread,
            avg(depth_ratio) as depth,
            nanstd(log_ret) as intra_vol,

            COUNT(*) as bar_count

        FROM cte_rolling
        GROUP BY instrument, trading_day
        HAVING bar_count > 10
    )

    SELECT
        CAST(trading_day AS DATETIME) AS date, instrument,
        open_p, high_p, low_p, close_p, total_volume,
        adjusted_pressure, imb_mean, imb_close, imb_std,
        spread, depth, intra_vol
    FROM cte_window
    ORDER BY instrument, date
    """

    df = dai.query(
        sql,
        filters={'date': [query_start_date.strftime('%Y-%m-%d'), end_date]},
        compression=True,
    ).df()

    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)
    g = df.groupby('instrument')

    # ================================================================
    # 三因子体系 (统一方向: 反转/低风险/均值回归)
    # ================================================================
    df['ret']     = g['close_p'].pct_change().fillna(0)
    df['ret_on']  = df['open_p'] / g['close_p'].shift(1) - 1.0
    df['ampl']    = (df['high_p'] - df['low_p']) / df['close_p'].clip(1e-8)
    df['turn']    = df['total_volume'] / g['total_volume'].transform(
        lambda x: x.rolling(120, min_periods=60).mean()).clip(1e-8)
    df['illiq']   = df['ret'].abs() / (df['total_volume'] * df['close_p']).clip(1e-8)
    df['vshock']  = df['total_volume'] / g['total_volume'].transform(
        lambda x: x.rolling(20, min_periods=10).mean()).clip(1e-8)

    def roll(col, w, agg='mean'):
        if agg == 'std':
            return g[col].transform(lambda x: x.rolling(w, min_periods=max(3,w//2)).std())
        if agg == 'max':
            return g[col].transform(lambda x: x.rolling(w, min_periods=max(3,w//2)).max())
        return g[col].transform(lambda x: x.rolling(w, min_periods=max(3,w//2)).mean())

    # --- Factor 1: 反转 (A股最强, 40%) ---
    f_rev  = -g['ret'].shift(1) * 0.15           # 1日反转
    f_rev += -roll('ret', 5) * 0.10               # 5日反转
    f_rev += -roll('ret', 21) * 0.05              # 21日反转
    f_rev += -g['ret_on'].shift(1) * 0.10         # 隔夜反转

    # --- Factor 2: 低波动 + 流动性 (30%) ---
    f_low  = -roll('ret', 5, 'std') * 0.08        # 低波5日
    f_low += -roll('ret', 21, 'std') * 0.08       # 低波21日
    f_low += -df['ampl'] * 0.04                     # 低振幅
    f_low += -roll('ampl', 21) * 0.03              # 低振幅趋势
    f_low += -roll('turn', 5) * 0.04               # 低换手
    f_low += -df['vshock'] * 0.03                   # 非异常放量

    # --- Factor 3: 微观结构反转 (30%) ---
    f_mic  = -df['adjusted_pressure'] * 0.08      # 压力反转(模板核心)
    f_mic += -df['imb_close'] * 0.05              # 收盘不平衡反转
    f_mic += -roll('imb_mean', 5) * 0.04          # 持续买压反转
    f_mic +=  roll('illiq', 21) * 0.03            # 非流动性溢价
    f_mic +=  df['spread'] * 0.03                  # 价差补偿
    f_mic += -roll('intra_vol', 5) * 0.04          # 低日内波动

    # ================================================================
    # 横截面 z-score + 等权合成
    # ================================================================
    def cs_zscore(vals):
        a = np.asarray(vals, dtype=np.float64)
        out = np.full(a.shape, np.nan)
        ok = ~np.isnan(a)
        if ok.sum() < 10:
            return out
        mu, sd = a[ok].mean(), a[ok].std()
        w = a[ok].clip(mu - 3.5*sd, mu + 3.5*sd)
        mu2, sd2 = w.mean(), w.std()
        if sd2 < 1e-10:
            return out
        out[ok] = (w - mu2) / sd2
        return out

    df['_f_rev'] = f_rev
    df['_f_low'] = f_low
    df['_f_mic'] = f_mic

    df['z_rev'] = df.groupby('date')['_f_rev'].transform(cs_zscore)
    df['z_low'] = df.groupby('date')['_f_low'].transform(cs_zscore)
    df['z_mic'] = df.groupby('date')['_f_mic'].transform(cs_zscore)

    df['raw'] = (df['z_rev'].fillna(0) * 0.40 +
                 df['z_low'].fillna(0) * 0.30 +
                 df['z_mic'].fillna(0) * 0.30)

    # 三个全缺设 NaN
    all_bad = df['z_rev'].isna() & df['z_low'].isna() & df['z_mic'].isna()
    df.loc[all_bad, 'raw'] = np.nan

    # ================================================================
    # AR(1) 平滑 + 最终截面 z-score
    # ================================================================
    df['smooth'] = g['raw'].transform(
        lambda x: x.ewm(alpha=0.88, min_periods=1).mean())
    df['factor'] = df.groupby('date')['smooth'].transform(cs_zscore)

    # ================================================================
    # 股票池对齐
    # ================================================================
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, pool, how='inner', on=['date', 'instrument'])

    req_start = pd.to_datetime(start_date)
    df = df[(df['date'] >= req_start) & df['factor'].notna()].copy()
    return df[['date', 'instrument', 'factor']].reset_index(drop=True)


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog
    logger = structlog.get_logger()
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-06-30 09:08:11] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
